# Naive Bayes 텍스트 분류

Naive Bayes는 문서에 나타난 단어의 빈도를 이용해 문서가 어떤 클래스에서 만들어졌을 가능성이 큰지 계산하는 확률 기반 분류 모델이다.

텍스트를 단어 빈도나 TF-IDF 벡터로 바꾸면 복잡한 신경망 없이도 빠르게 학습할 수 있어 텍스트 분류의 기준 모델로 널리 사용한다. 스팸 탐지, 뉴스 주제 분류, 감성 분류처럼 단어 선택이 클래스와 밀접한 문제에서 특히 유용하다.

이번 교안은 작은 문장으로 확률의 의미를 먼저 확인한 뒤, 20 Newsgroups 뉴스 문서를 실제로 학습한다. 최빈 클래스만 고르는 기준선과 비교하고, 예측확률·클래스별 주요 단어·오분류 문서를 읽어 모델이 무엇을 배웠는지 해석한다.

## 01. 단어 빈도에서 클래스 확률까지

**Multinomial Naive Bayes**는 클래스마다 어떤 단어가 자주 등장하는지를 학습한 뒤, 새 문장의 단어들이 각 클래스에서 나타날 확률을 결합한다.
이름의 `Naive`는 문서의 클래스가 정해졌을 때 각 단어가 서로 독립이라고 단순하게 가정한다는 뜻이다.

이 가정은 실제 언어를 완벽하게 설명하지 못하지만, 희소한 단어 빈도 벡터에서 계산이 빠르고 데이터가 많지 않아도 비교적 안정적인 기준 성능을 만든다.

- **`CountVectorizer`**: 문서 목록을 문서별 단어 빈도 행렬로 변환하는 도구
- **특성 우도(Feature Likelihood)**: 특정 클래스에서 각 단어가 나타날 확률
- **사후확률(Posterior Probability)**: 문서를 관찰한 뒤 각 클래스일 확률
- **`alpha`**: 보지 못한 단어 조합의 확률이 0이 되지 않도록 빈도를 보정하는 값


### 작은 문장을 문서×단어 빈도 행렬로 변환

세 개의 학습 문장에는 `긍정` 또는 `부정` 정답을 붙인다. `CountVectorizer`가 만든 행렬의 한 행은 문서 하나이고, 한 열은 어휘 하나이며, 각 값은 그 문서에서 단어가 등장한 횟수이다.


In [1]:
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, classification_report, f1_score

toy_texts = [
    "재미 감동 추천",
    "재미 유쾌 추천",
    "지루 실망 비추천",
    "지루 최악 비추천",
]
toy_labels = np.array(["긍정", "긍정", "부정", "부정"])

toy_vectorizer = CountVectorizer()
# 학습 문장에서 어휘를 만들고 문장×단어 빈도 행렬로 변환한다.
toy_counts = toy_vectorizer.fit_transform(toy_texts)

toy_words = toy_vectorizer.get_feature_names_out()

print("어휘 순서:", toy_words.tolist())
print("빈도 행렬 shape:", toy_counts.shape)
print("빈도 행렬:")
print(toy_counts.toarray())



어휘 순서: ['감동', '비추천', '실망', '유쾌', '재미', '지루', '최악', '추천']
빈도 행렬 shape: (4, 8)
빈도 행렬:
[[1 0 0 0 1 0 0 1]
 [0 0 0 1 1 0 0 1]
 [0 1 1 0 0 1 0 0]
 [0 1 0 0 0 1 1 0]]


### 클래스별 단어 분포 학습과 새 문장 예측


In [2]:
toy_model = MultinomialNB(alpha=1.0)
# 빈도 행렬과 정답을 연결해 클래스별 단어 분포를 학습한다.
toy_model.fit(toy_counts, toy_labels)

new_texts = ["재미 추천", "지루 비추천", "재미 지루"]
# 새 문장은 학습된 어휘 순서를 유지하도록 transform만 적용한다.
new_counts = toy_vectorizer.transform(new_texts)

# 클래스별 확률과 가장 큰 확률의 예측 클래스를 함께 구한다.
probabilities = toy_model.predict_proba(new_counts)  # 각 클래스 확률  -> predict_proba
predictions = toy_model.predict(new_counts)  # 예측 결과(긍정/부정)  -> predict

print("클래스 열 순서:", toy_model.classes_.tolist())
for text, prediction, probability in zip(new_texts, predictions, probabilities):
    class_probability = {
        str(label): round(float(score), 3)
        for label, score in zip(toy_model.classes_, probability)
    }
    print(f"{text!r} → 예측={prediction}, 확률={class_probability}")



클래스 열 순서: ['긍정', '부정']
'재미 추천' → 예측=긍정, 확률={'긍정': 0.9, '부정': 0.1}
'지루 비추천' → 예측=부정, 확률={'긍정': 0.1, '부정': 0.9}
'재미 지루' → 예측=긍정, 확률={'긍정': 0.5, '부정': 0.5}


## 02. 실제 뉴스 주제 분류

20 Newsgroups는 여러 뉴스그룹의 영문 게시글을 주제별로 모은 데이터이다. 이번에는 `하키`, `우주`, `정치` 세 범주만 사용한다.

학습 데이터로만 TF-IDF 어휘와 가중치를 학습하고 테스트 데이터에는 `transform()`만 적용한다. 테스트 문서까지 `fit()`에 포함하면 평가 데이터의 단어 정보를 미리 본 데이터 누수가 발생한다.


### 학습 데이터와 테스트 데이터 불러오기

헤더·서명·인용문을 제거하여 작성자나 뉴스그룹 이름 같은 쉬운 단서보다 본문 단어로 주제를 구분하게 한다. `target_names`의 순서가 정수 레이블의 의미이다.


In [3]:
# 하키, 우주, 정치  3개 카테고리만 사용

categories = ["rec.sport.hockey", "sci.space", "talk.politics.misc"]

# 학습용과 테스트용 문서를 분리해 평가 데이터 누수를 막는다.
news_train = fetch_20newsgroups(
    subset="train", # 학습용 데이터
    categories=categories, # 하키, 우주, 정치 카테고리 뉴스만 다운로드
    remove=("headers", "footers", "quotes"),
)
news_test = fetch_20newsgroups(
    subset="test",
    categories=categories,
    remove=("headers", "footers", "quotes"),
)

print("클래스 ID 순서:", list(enumerate(news_train.target_names)))
print("학습 문서 수:", len(news_train.data))
print("테스트 문서 수:", len(news_test.data))
print("학습 클래스별 문서 수:", np.bincount(news_train.target).tolist())


클래스 ID 순서: [(0, 'rec.sport.hockey'), (1, 'sci.space'), (2, 'talk.politics.misc')]
학습 문서 수: 1658
테스트 문서 수: 1103
학습 클래스별 문서 수: [600, 593, 465]


### TF-IDF 변환, 최빈 클래스 기준선과 Naive Bayes 학습

TF-IDF는 한 문서에서는 자주 등장하지만 모든 문서에 흔하지는 않은 단어에 더 큰 값을 준다.
`DummyClassifier`는 문장 내용을 보지 않고 가장 많은 클래스만 예측하므로 실제 모델이 넘어야 할 최소 기준선이다.

Naive Bayes의 `alpha`는 평활 강도이다. 이 값이 지나치게 크면 클래스별 단어 차이가 약해지고, 지나치게 작으면 드문 단어에 민감해질 수 있다.

다음 빈 코드셀에서 TF-IDF 변환, 기준선과 Naive Bayes 학습을 완성해야 뒤의 성능 평가와 주요 단어 분석을 실행할 수 있다.

In [4]:
tfidf = TfidfVectorizer(
    stop_words="english", # 영어 불용어 처리
    ngram_range=(1,2), # 단어 하나와 연속된 두 단어 모두를 특성 후보로 사용
    max_features=12_000, # feature 개수를 12000개로 제한
    min_df=2, # 문서에서 단어가 2번 이상 나와야 특성(feature)에 추가.
    sublinear_tf=True # 반복횟수 tf를 '1+log(tf)'로 줄여 매우 자주 등장하는 단어의 영향력을 줄임
)

X_train = tfidf.fit_transform(news_train.data)
X_test = tfidf.transform(news_test.data)

# 베이스라인용 더미모델 생성
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, news_train.target)

# MultinomialNB 모델을 생성하여 TF-IDF 특성 분포와 클래스 학습
# alpha=0.01 : 특성에 작은 값을 더해서 클래스별 단어 차이를 비교적 선명하게 유지한다.
news_model = MultinomialNB(alpha=0.01)
news_model.fit(X_train, news_train.target)

print('학습 TF-IDF shape', X_train.shape)
print('평가 TF-IDF shape', X_test.shape)
print('모델에 학습된 클래스 수', len(news_model.classes_))


학습 TF-IDF shape (1658, 12000)
평가 TF-IDF shape (1103, 12000)
모델에 학습된 클래스 수 3


### 기준선과 테스트 성능 비교

accuracy는 전체 문서 중 맞힌 비율이고, macro F1은 클래스별 F1을 같은 비중으로 평균낸 값이다. 클래스 수가 불균형할 수 있으므로 두 지표를 함께 확인한다.


In [5]:
# 같은 테스트 행렬로 기준선과 학습 모델의 예측을 구한다.
baseline_predictions = baseline.predict(X_test)
news_predictions = news_model.predict(X_test)

baseline_accuracy = accuracy_score(news_test.target, baseline_predictions)
# macro F1은 클래스별 F1을 같은 비중으로 평균한다.
baseline_macro_f1 = f1_score(
    news_test.target,
    baseline_predictions,
    average="macro",
)

model_accuracy = accuracy_score(news_test.target, news_predictions)
model_macro_f1 = f1_score(
    news_test.target,
    news_predictions,
    average="macro",
)

print(f"최빈 클래스 기준선 accuracy: {baseline_accuracy:.3f}")
print(f"최빈 클래스 기준선 macro F1: {baseline_macro_f1:.3f}")
print(f"Naive Bayes accuracy: {model_accuracy:.3f}")
print(f"Naive Bayes macro F1: {model_macro_f1:.3f}")
print()
print(
    classification_report(
        news_test.target,
        news_predictions,
        target_names=news_train.target_names,
        digits=3,
    )
)



최빈 클래스 기준선 accuracy: 0.362
최빈 클래스 기준선 macro F1: 0.177
Naive Bayes accuracy: 0.906
Naive Bayes macro F1: 0.903

                    precision    recall  f1-score   support

  rec.sport.hockey      0.909     0.952     0.930       399
         sci.space      0.935     0.876     0.904       394
talk.politics.misc      0.867     0.884     0.875       310

          accuracy                          0.906      1103
         macro avg      0.904     0.904     0.903      1103
      weighted avg      0.907     0.906     0.906      1103



### 모델이 학습한 주요 단어와 오분류 문서

`feature_log_prob_`의 한 행은 한 클래스에서 각 TF-IDF 특성이 나타날 로그확률이다. 값이 큰 단어를 보면 모델이 각 주제를 어떤 표현과 연결했는지 확인할 수 있다.

오분류 문서는 숫자 지표가 감추는 문제를 보여 준다. 본문이 짧거나 여러 주제를 함께 다루거나 핵심 단어가 제거된 문서는 잘못 분류될 가능성이 크다.


In [6]:
feature_names = tfidf.get_feature_names_out()

for class_id, class_name in enumerate(news_train.target_names):
    # 클래스 행의 로그확률을 꺼내 값이 큰 단어를 찾는다.
    class_log_probabilities = news_model.feature_log_prob_[class_id]

    sorted_indices = class_log_probabilities.argsort()
    # 뒤의 10개 인덱스를 선택하고 큰 값부터 읽도록 뒤집는다.
    top_indices = sorted_indices[-10:][::-1]
    top_words = feature_names[top_indices].tolist()
    print(f"{class_name} 주요 단어: {top_words}")

# 예측과 정답이 다른 테스트 행의 위치만 뽑는다.
wrong_indices = np.flatnonzero(news_predictions != news_test.target)
nonempty_wrong_indices = [
    index
    for index in wrong_indices
    if news_test.data[index].strip()
]
print(f"\n오분류 문서 수: {len(wrong_indices)} / {len(news_test.target)}")

for index in nonempty_wrong_indices[:3]:
    preview = " ".join(news_test.data[index].split())[:220]

    actual = news_train.target_names[news_test.target[index]]
    predicted = news_train.target_names[news_predictions[index]]

    # 한 문서의 클래스 확률 행을 꺼내 레이블과 연결한다.
    document_probabilities = news_model.predict_proba(X_test[index])[0]
    class_probabilities = {
        news_train.target_names[predicted_class_id]: round(float(score), 3)
        for predicted_class_id, score in zip(news_model.classes_, document_probabilities)
    }

    print(f"\n정답={actual} / 예측={predicted}")
    print("클래스별 확률:", class_probabilities)
    print("본문:", preview)



rec.sport.hockey 주요 단어: ['game', 'team', 'hockey', 'play', 'games', 'season', 'players', 'nhl', 'year', 'think']
sci.space 주요 단어: ['space', 'nasa', 'like', 'just', 'orbit', 'launch', 'moon', 'think', 'earth', 'know']
talk.politics.misc 주요 단어: ['people', 'don', 'government', 'just', 'think', 'know', 'clinton', 'tax', 'like', 'state']

오분류 문서 수: 104 / 1103

정답=sci.space / 예측=talk.politics.misc
클래스별 확률: {'rec.sport.hockey': 0.112, 'sci.space': 0.28, 'talk.politics.misc': 0.608}
본문: Using greenhouses to extend the growing season shouldn't be a problem. I'm supprised they don't do so in Alaska (cheaper to import, perhaps?) No, the Incas had no problems with this, but the Spanish did.

정답=talk.politics.misc / 예측=rec.sport.hockey
클래스별 확률: {'rec.sport.hockey': 0.732, 'sci.space': 0.224, 'talk.politics.misc': 0.044}
본문: garrod@dynamo.ecn.purdue.edu (David Garrod) writes... It was on CBS yesterday. The explanation is reasonable enough. Then again, if the fire was accidental, why didn't more peop

## 정리와 한계

Naive Bayes는 `문서 → 단어 특성 → 클래스별 단어 확률 → 예측확률`의 흐름으로 텍스트를 분류한다. 학습과 예측이 빠르고 판단에 사용한 주요 단어를 확인하기 쉬워 복잡한 모델을 적용하기 전 기준선을 만드는 데 적합하다.

하지만 단어 순서와 긴 문맥을 직접 표현하지 못하고, 독립 가정 때문에 부정 표현이나 문장 구조를 놓칠 수 있다. 다음 교안에서는 토큰 순서를 읽는 LSTM과 지역 표현을 찾는 TextCNN을 학습하여 이 차이를 확인한다.
